In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import nltk
import random

# Assicurati di avere i token di NLTK per il tokenization
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\066321758\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:

# Caricamento del testo (usa un libro dell'autore che desideri)
with open('testo_autore.txt', 'r', encoding='utf-8') as file:
    text = file.read().lower()

# Tokenizzazione del testo
tokens = nltk.word_tokenize(text)

# Creazione del vocabolario unico (mappa le parole in indici)
vocab = sorted(set(tokens))
vocab_size = len(vocab)
word_to_int = {word: i for i, word in enumerate(vocab)}
int_to_word = {i: word for i, word in enumerate(vocab)}

# Creazione delle sequenze di input (X) e target (y)
seq_length = 40  # Numero di parole in ciascuna sequenza
sequences = []

for i in range(seq_length, len(tokens)):
    seq = tokens[i-seq_length:i]  # Sequenza di 40 parole
    label = tokens[i]  # Parola successiva
    sequences.append([word_to_int[word] for word in seq] + [word_to_int[label]])

# Creazione degli input (X) e degli output (y)
sequences = np.array(sequences)
X, y = sequences[:, :-1], sequences[:, -1]


In [ ]:
# ============================================================
# CREAZIONE DEL MODELLO LSTM
# ============================================================

# Creiamo una rete neurale sequenziale.
#
# Sequential permette di organizzare i diversi livelli
# della rete uno dopo l'altro, seguendo un ordine preciso.
model = Sequential()


# ------------------------------------------------------------
# 1. EMBEDDING
# ------------------------------------------------------------

# Il livello Embedding converte gli indici numerici delle parole
# in vettori numerici più ricchi di informazioni.
#
# Parametri:
# - vocab_size: numero totale di parole presenti nel vocabolario.
# - 100: dimensione del vettore associato a ogni parola.
# - input_length=seq_length-1: lunghezza della sequenza in ingresso.
#
# ATTENZIONE:
# Nel codice precedente X contiene 40 parole per ogni sequenza.
# In questo caso, per mantenere la stessa lunghezza degli input,
# sarebbe più coerente usare input_length=seq_length.
model.add(
    Embedding(
        vocab_size,
        100,
        input_length=seq_length
    )
)


# ------------------------------------------------------------
# 2. PRIMO LIVELLO LSTM
# ------------------------------------------------------------

# Il primo livello LSTM analizza la sequenza di parole
# e cerca di imparare le relazioni tra le parole precedenti.
#
# 256 indica il numero di unità LSTM presenti nel livello.
# Un numero maggiore di unità permette al modello di imparare
# relazioni più complesse, ma richiede anche più risorse.
#
# return_sequences=True significa che il livello LSTM
# restituisce un risultato per ogni parola della sequenza.
#
# Questo è necessario perché dopo ci sarà un secondo livello LSTM
# che dovrà ricevere l'intera sequenza di risultati.
model.add(
    LSTM(
        256,
        return_sequences=True
    )
)


# ------------------------------------------------------------
# 3. DROPOUT
# ------------------------------------------------------------

# Il Dropout serve a ridurre il rischio di overfitting.
#
# Durante l'addestramento, il 20% delle connessioni viene
# temporaneamente ignorato in modo casuale.
#
# In questo modo il modello non impara semplicemente a memoria
# gli esempi del testo di addestramento, ma cerca di imparare
# schemi e relazioni più generali.
#
# 0.2 corrisponde al 20%.
model.add(
    Dropout(0.2)
)


# ------------------------------------------------------------
# 4. SECONDO LIVELLO LSTM
# ------------------------------------------------------------

# Il secondo livello LSTM riceve le informazioni elaborate
# dal primo livello e le analizza ulteriormente.
#
# Anche questo livello contiene 256 unità LSTM.
#
# In questo caso non impostiamo return_sequences=True.
# Di conseguenza, il livello restituisce soltanto il risultato
# finale dell'intera sequenza.
#
# Questo risultato riassume le informazioni più importanti
# contenute nelle parole analizzate.
model.add(
    LSTM(256)
)


# ------------------------------------------------------------
# 5. LIVELLO DENSE DI OUTPUT
# ------------------------------------------------------------

# Il livello Dense produce la previsione finale.
#
# vocab_size indica che il modello deve produrre un valore
# di probabilità per ogni parola presente nel vocabolario.
#
# activation='softmax' trasforma i valori prodotti dalla rete
# in probabilità comprese tra 0 e 1.
#
# La somma di tutte le probabilità è pari a 1.
#
# La parola con la probabilità più alta sarà quella
# scelta come parola successiva prevista.
model.add(
    Dense(
        vocab_size,
        activation='softmax'
    )
)


# ============================================================
# COMPILAZIONE DEL MODELLO
# ============================================================

# Prepariamo il modello per l'addestramento.
#
# loss='categorical_crossentropy':
# misura la differenza tra la parola corretta e la previsione
# prodotta dal modello.
#
# optimizer='adam':
# aggiorna i pesi della rete per ridurre progressivamente
# l'errore durante l'addestramento.
#
# metrics=['accuracy']:
# permette di visualizzare la precisione del modello,
# cioè quante volte la parola prevista corrisponde
# alla parola corretta.
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


# ============================================================
# PREPARAZIONE DEL TARGET: ONE-HOT ENCODING
# ============================================================

# In precedenza, y conteneva un solo indice numerico
# per ogni parola corretta.
#
# Esempio:
# y = [4, 2, 7]
#
# Con to_categorical(), ogni indice viene trasformato
# in un vettore one-hot.
#
# Se il vocabolario contiene 10 parole:
#
# 4 diventa:
# [0, 0, 0, 0, 1, 0, 0, 0, 0, 0]
#
# num_classes=vocab_size indica il numero totale
# di possibili parole da prevedere.
#
# Il risultato viene assegnato nuovamente a y.
y = to_categorical(
    y,
    num_classes=vocab_size
)


# ============================================================
# ADDESTRAMENTO DEL MODELLO
# ============================================================

# Avviamo l'addestramento della rete LSTM.
#
# X contiene le sequenze di parole precedenti:
# sono gli INPUT del modello.
#
# y contiene la parola successiva corretta:
# è il TARGET, cioè la risposta che il modello deve imparare
# a prevedere.
#
# epochs=50:
# il modello analizza l'intero insieme di dati 50 volte.
#
# batch_size=128:
# durante ogni aggiornamento, il modello utilizza
# 128 esempi alla volta, quando sono disponibili.
#
# Durante l'addestramento il modello:
# 1. riceve una sequenza di parole da X;
# 2. produce una previsione;
# 3. confronta la previsione con la parola corretta in y;
# 4. calcola l'errore;
# 5. aggiorna i propri pesi;
# 6. ripete il processo per molte sequenze.
#
# In questo modo impara a prevedere la parola successiva
# osservando le parole che la precedono.
model.fit(
    X,
    y,
    epochs=50,
    batch_size=128
)

Epoch 1/50


C:\Users\ANGELAANGUILANO\simple-environment\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


11/11 ━━━━━━━━━━━━━━━━━━━━ 12s 541ms/step - accuracy: 0.0365 - loss: 5.9359
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 542ms/step - accuracy: 0.0447 - loss: 5.3284
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 539ms/step - accuracy: 0.0529 - loss: 5.2133
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 512ms/step - accuracy: 0.0455 - loss: 5.1867
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 482ms/step - accuracy: 0.0484 - loss: 5.1744
Epoch 6/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 5s 492ms/step - accuracy: 0.0484 - loss: 5.1718
Epoch 7/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 496ms/step - accuracy: 0.0425 - loss: 5.1698
Epoch 8/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 481ms/step - accuracy: 0.0492 - loss: 5.1676
Epoch 9/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 505ms/step - accuracy: 0.0581 - loss: 5.1614
Epoch 10/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 9s 442ms/step - accuracy: 0.0499 - loss: 5.1582
Epoch 11/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 5s 471ms/step - accuracy: 0.0626 - loss: 5.1508
Epoch 12/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 11s 513ms/step - acc

In [4]:

# Funzione per generare del testo
def generate_text(model, int_to_word, word_to_int, seed_text, seq_length, num_words):
    # Inizializza il testo con il seed_text
    result = seed_text
    input_text = seed_text.lower().split()
    
    # Genera 'num_words' parole
    for _ in range(num_words):
        # Converte il testo in sequenza numerica
        encoded = [word_to_int[word] for word in input_text if word in word_to_int]
        
        # Padding per la sequenza (poiché la sequenza di input deve avere una lunghezza fissa)
        encoded = pad_sequences([encoded], maxlen=seq_length-1, padding='pre')
        
        # Predice la parola successiva
        pred_probs = model.predict(encoded, verbose=0)
        predicted_idx = np.argmax(pred_probs)
        
        # Converte l'indice predetto in parola
        predicted_word = int_to_word[predicted_idx]
        
        # Aggiungi la parola predetta al testo
        result += ' ' + predicted_word
        
        # Aggiungi la parola predetta alla sequenza di input per la prossima previsione
        input_text.append(predicted_word)
        input_text = input_text[1:]
    
    return result


In [5]:
# Genera un testo nello stile dell'autore
seed_text = "era una notte buia"  # Testo iniziale
generated_text = generate_text(model, int_to_word, word_to_int, seed_text, seq_length, 100)
print(generated_text)

era una notte buia con la tuo partner con te . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo capoufficio . è il tuo
